# 2.1 Setup and Dependencies

In [1]:
# Install required libraries
!pip install llama-index fastembed kdbai_client onnxruntime==1.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.1/233.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/1

In [3]:
import os
from getpass import getpass
import kdbai_client as kdbai
import time
from llama_index.core import Document, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
import pandas as pd
from fastembed import TextEmbedding
import openai
import textwrap

# 2.2 Connecting to KDB.AI

In [4]:
KDBAI_ENDPOINT="KDBAI_ENDPOINT"
KDBAI_API_KEY="KDBAI_API_KEY"

os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"
fastembed = TextEmbedding()
KDBAI_TABLE_NAME = "paul_graham"
session = kdbai.Session(endpoint=KDBAI_ENDPOINT, api_key=KDBAI_API_KEY)
database = session.database("default")
# Drop table if exists
try:
    database.table(KDBAI_TABLE_NAME).drop()
except kdbai.KDBAIException:
    pass
schema = [
    dict(name="text", type="bytes"),
    dict(name="embedding", type="float32s")
]
indexes = [dict(name="flat_index", column="embedding", type="flat", params=dict(metric="L2", dims=384))]
table = database.create_table(KDBAI_TABLE_NAME, schema=schema, indexes=indexes)

In [ ]:
fastembed = TextEmbedding()

# 2.3 Data Prep: Paul Graham Essays

In [5]:
!mkdir -p ./data
!llamaindex-cli download-llamadataset PaulGrahamEssayDataset --download-dir ./data

node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=100)
essays = SimpleDirectoryReader(input_dir="./data/source_files").load_data()
docs = node_parser.get_nodes_from_documents(essays)
len(docs)

In [6]:
embedding_model = TextEmbedding()
documents = [doc.text for doc in docs]
embeddings = list(embedding_model.embed(documents))

records_to_insert_with_embeddings = pd.DataFrame({
    "text": [d.encode('utf-8') for d in documents],
    "embedding": embeddings
})

table.insert(records_to_insert_with_embeddings)

# 2.4 RAG Implementation

In [7]:
query = "How does Paul Graham decide what to work on?"
query_embedding = list(embedding_model.embed([query]))[0].tolist()

search_results = table.search({"flat_index": [query_embedding]}, n=10)
search_results_df = search_results[0]
df = pd.DataFrame(search_results_df)
df.head(5)

# 2.5 The Citation Pipeline Code

In [9]:
#!/usr/bin/env python3

import os
import re
import json
import openai
import pandas as pd
from typing import List, Dict, Any
from IPython.display import display, HTML

################################################################################
# STEP 1: PREPARE DATA
################################################################################

def parse_chunk_into_sentences(chunk_text: str) -> List[Dict[str, Any]]:
    """
    Splits 'chunk_text' into naive 'sentences' with start/end offsets.
    Returns a list of dicts like:
      {
        "sentence_id": int,
        "text": str,
        "start_char": int,
        "end_char": int
      }
    """
    # We'll do a simple regex to split on '.' while capturing the period if it appears
    # Then we re-join it. A robust approach might use spacy or NLTK, but for demonstration:
    import re
    raw_parts = re.split(r'(\.)', chunk_text)

    # We'll combine text + punctuation
    combined = []
    for i in range(0, len(raw_parts), 2):
        text_part = raw_parts[i].strip()
        punct = ""
        if i+1 < len(raw_parts):
            punct = raw_parts[i+1]
        if text_part or punct:
            combined_text = (text_part + punct).strip()
            if combined_text:
                combined.append(combined_text)

    sentences = []
    offset = 0
    for s_id, s_txt in enumerate(combined, start=1):
        start_char = offset
        end_char = start_char + len(s_txt)
        sentences.append({
            "sentence_id": s_id,
            "text": s_txt,
            "start_char": start_char,
            "end_char": end_char
        })
        offset = end_char + 1  # assume space or newline after each
    return sentences

################################################################################
# STEP 2: CALL OPENAI WITH A ROBUST SYSTEM PROMPT
################################################################################

def call_openai_with_citations(chunks: List[str], user_query: str) -> str:
    """
    Asks the LLM to produce a single continuous answer,
    referencing chunk_id + sentences range as:
      <CIT chunk_id='N' sentences='X-Y'>...some snippet...</CIT>.
    """

    # If you want, set your API key in code or rely on environment variable
    # openai.api_key = "sk-..."
    if not openai.api_key and "OPENAI_API_KEY" in os.environ:
        openai.api_key = os.environ["OPENAI_API_KEY"]

    # We'll craft a robust system prompt with examples
    system_prompt = (
        "You have a collection of chunks from a single document, each chunk may have multiple sentences.\n"
        "Please write a single continuous answer to the user's question.\n"
        "When you reference or rely on a specific portion of a chunk, cite it as:\n"
        "  <CIT chunk_id='N' sentences='X-Y'>the snippet of your final answer</CIT>\n"
        "Where:\n"
        "  - N is the chunk index.\n"
        "  - X-Y is the range of sentence numbers within that chunk. Example: 'sentences=2-4'.\n"
        "  - The text inside <CIT> is part of your answer, not the original chunk text.\n"
        "  - Keep your answer minimal in whitespace. Do not add extra spaces or line breaks.\n"
        "  - Only add <CIT> tags around the key phrases of your answer that rely on some chunk.\n"
        "    E.g. 'He stated <CIT chunk_id='3' sentences='1-2'>it was crucial to experiment early</CIT>.'\n\n"
        "Remember: The text inside <CIT> is your final answer's snippet, not the chunk text itself.\n"
        "The user question is below."
    )

    # We just show the user the chunk texts:
    chunks_info = "\n\n".join(
        f"[Chunk {i}] {chunk}" for i, chunk in enumerate(chunks)
    )

    # We create the conversation
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"{chunks_info}\n\nQuestion: {user_query}\n"
        }
    ]

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.3,
        max_tokens=1024
    )
    return response.choices[0].message.content

################################################################################
# STEP 3: PARSE THE LLM RESPONSE
################################################################################

def parse_response_with_sentence_range(response_text: str) -> Dict[str, Any]:
    """
    Produce a single block with:
    {
      "type": "text",
      "text": <the final answer minus CIT tags but with snippet inline>,
      "citations": [
        {
          "chunk_id": int,
          "sentences_range": "X-Y",
          "answer_snippet": snippet,
          "answer_snippet_start": int,
          "answer_snippet_end": int
        },
        ...
      ]
    }
    """
    pattern = re.compile(
        r'(.*?)<CIT\s+chunk_id=[\'"](\d+)[\'"]\s+sentences=[\'"](\d+-\d+)[\'"]>(.*?)(?:</CIT>|(?=<CIT)|$)',
        re.DOTALL
    )
    final_text = ""
    citations = []
    idx = 0

    while True:
        match = pattern.search(response_text, idx)
        if not match:
            # leftover
            leftover = response_text[idx:]
            final_text += leftover
            break

        text_before = match.group(1)
        chunk_id_str = match.group(2)
        sent_range = match.group(3)
        snippet = match.group(4)

        final_text += text_before

        start_in_answer = len(final_text)
        final_text += snippet
        end_in_answer = len(final_text)

        citations.append({
            "chunk_id": int(chunk_id_str),
            "sentences_range": sent_range,
            "answer_snippet": snippet,
            "answer_snippet_start": start_in_answer,
            "answer_snippet_end": end_in_answer
        })

        idx = match.end()

    return {
        "type": "text",
        "text": final_text,
        "citations": citations
    }

################################################################################
# STEP 4: MATCH CITED SENTENCES + FIND CHAR RANGES IN CHUNK
################################################################################

def gather_sentence_data_for_citations(block: Dict[str, Any], sentence_map: Dict[int, List[Dict[str, Any]]]) -> Dict[str, Any]:
    """
    For each citation, parse the chunk_id + sentences='X-Y'.
    Gather the text of those sentences from 'sentence_map[chunk_id]'
    and record their combined text plus start/end offsets in the chunk.
    """
    for c in block["citations"]:
        c_id = c["chunk_id"]
        sent_range = c["sentences_range"]
        try:
            start_sent, end_sent = map(int, sent_range.split("-"))
        except:
            start_sent, end_sent = 1, 1

        # get the sentence list for that chunk
        sents_for_chunk = sentence_map.get(c_id, [])
        # filter the range
        relevant_sents = [s for s in sents_for_chunk if start_sent <= s["sentence_id"] <= end_sent]

        if relevant_sents:
            combined_text = " ".join(s["text"] for s in relevant_sents)
            chunk_start_char = relevant_sents[0]["start_char"]
            chunk_end_char = relevant_sents[-1]["end_char"]
        else:
            combined_text = ""
            chunk_start_char = -1
            chunk_end_char = -1

        c["chunk_sentences_text"] = combined_text
        c["chunk_sentences_start"] = chunk_start_char
        c["chunk_sentences_end"] = chunk_end_char

    return block

################################################################################
# STEP 5: BUILD HTML FOR DISPLAY
################################################################################

def build_html_for_block(block: Dict[str, Any]) -> str:
    """
    Build an HTML string that underlines each snippet in the final answer
    and shows a tooltip with 'chunk_sentences_text' plus start/end offsets.
    """
    css = """
    <style>
    body {
      font-family: Arial, sans-serif;
      margin: 20px;
      line-height: 1.6;
    }
    .tooltip {
      position: relative;
      text-decoration: underline dotted;
      cursor: help;
    }
    .tooltip .tooltiptext {
      visibility: hidden;
      width: 400px;
      background: #f9f9f9;
      color: #333;
      text-align: left;
      border: 1px solid #ccc;
      border-radius: 4px;
      padding: 10px;
      position: absolute;
      z-index: 1;
      top: 125%;
      left: 50%;
      transform: translateX(-50%);
      opacity: 0;
      transition: opacity 0.3s;
    }
    .tooltip:hover .tooltiptext {
      visibility: visible;
      opacity: 1;
    }
    </style>
    """

    full_text = block["text"]
    citations = sorted(block["citations"], key=lambda x: x["answer_snippet_start"])

    html_parts = [f"<!DOCTYPE html><html><head><meta charset='UTF-8'>{css}</head><body>"]
    cursor = 0

    for cit in citations:
        st = cit["answer_snippet_start"]
        en = cit["answer_snippet_end"]

        if st > cursor:
            html_parts.append(full_text[cursor:st])

        snippet_text = full_text[st:en]

        # Build tooltip with chunk sentences
        tooltip_html = f"""
        <span class="tooltip">
          {snippet_text}
          <span class="tooltiptext">
            <strong>Chunk ID:</strong> {cit["chunk_id"]}<br>
            <strong>Sentence Range:</strong> {cit["sentences_range"]}<br>
            <strong>Chunk Sentences Offset:</strong> {cit["chunk_sentences_start"]}-{cit["chunk_sentences_end"]}<br>
            <strong>Chunk Sentences Text:</strong> {cit["chunk_sentences_text"]}
          </span>
        </span>
        """
        html_parts.append(tooltip_html)
        cursor = en

    if cursor < len(full_text):
        html_parts.append(full_text[cursor:])

    html_parts.append("</body></html>")
    return "".join(html_parts)

def display_html_block(block: Dict[str, Any]):
    from IPython.display import display, HTML
    html_str = build_html_for_block(block)
    display(HTML(html_str))

################################################################################
# PUTTING IT ALL TOGETHER
################################################################################

def main(df, user_query: str):
    """
    Full pipeline:
      1) We'll parse each chunk into sentences.
      2) We'll call openai with a robust system prompt for <CIT> usage.
      3) We'll parse the LLM's response for chunk_id + sentences='X-Y'.
      4) We'll gather the chunk sentences text, produce a final block with citations.
      5) We'll build HTML and display in Colab.
    """

    # 1) Prepare chunk data
    # - We'll assume df has columns: chunk_id, text
    # - We'll parse each chunk into sentence_map
    sentence_map = {}
    chunk_texts = []
    max_chunk_id = df["chunk_id"].max()
    for i, row in df.iterrows():
        c_id = row["chunk_id"]
        c_txt = row["text"]
        # build the sentence parse
        sents = parse_chunk_into_sentences(c_txt)
        sentence_map[c_id] = sents
        # We'll store chunk_texts in an array in chunk_id order
        # If chunk_id is not sequential from 0..N, you might do a dict.
        # But let's do the simplest approach.
        # We'll expand chunk_texts if needed
        if len(chunk_texts) <= c_id:
            chunk_texts.extend([""]*(c_id - len(chunk_texts)+1))
        chunk_texts[c_id] = c_txt

    # 2) Call LLM
    answer_text = call_openai_with_citations(chunk_texts, user_query)

    # 3) Parse the response
    block = parse_response_with_sentence_range(answer_text)

    # 4) Enrich each citation with chunk sentences
    block = gather_sentence_data_for_citations(block, sentence_map)

    # 5) Display final result
    print("----- JSON OUTPUT -----")
    print(json.dumps({"content": [block]}, indent=2, ensure_ascii=False))

    display_html_block(block)


#############################
# Example usage in Colab
#############################
if __name__ == "__main__":
    # Suppose your df is already loaded with chunk_id, text columns
    # Convert all chunks to strings to prevent bytes errors
    chunks = [c.decode("utf-8", errors="replace") if isinstance(c, bytes) else str(c) for c in df["text"].tolist()]

    df_chunks = pd.DataFrame({"text": chunks})
    df_chunks["chunk_id"] = range(len(chunks))

    # The user query
    user_query = "How does Paul Graham decide what to work on?"

    main(df_chunks, user_query)

